# Sentinel-2 L2A: mosaicos y clasificación de fuego

Prepare Colab → cargue vectores → inspeccione GEE → descargue y compare → dibuje muestras → clasifique con RF.

Edite los parámetros aquí; la lógica está en `Sentinel2_pipeline_test.py`.
Las escenas se procesan en `/content/sentinel2_temporal`; el mosaico terminado se conserva en Drive.

## 1. Configuración y preparación de Colab

Edite las variables de la primera celda y ejecute ambas celdas en orden.
La segunda celda reemplaza el clon en `/content/fire_monitor`, instala dependencias y configura la autenticación de GEE/GCS mediante `M0_auth_config.py`.
Los títulos de las celdas permiten ocultar el código; los resultados se guardan fuera del clon.


In [ ]:
#@title 1. Variables del proyecto
COUNTRY_SELECTED = "colombia"
CAMPAIGN_SELECTED = "MONITOR_01"
MOSAIC_YEARS = [2025, 2026]

REPO_URL = "https://github.com/mapbiomas/colombia-fire.git"
REPO_BRANCH = "main"
REPO_DIR = "/content/fire_monitor"
CORE_RELATIVE_PATH = "mapbiomas_fire_monitor/version_01/src/core"
DIRECTORIO_TRABAJO = "/content/sentinel2_temporal"

# Proyecto y regiones utilizados actualmente por este notebook.
GEE_PROJECT = "mapbiomas-fire-485203"
ASSET_REGIONS = "projects/mapbiomas-colombia/assets/FIRE/AUXILIARY_DATA/regiones_monitor_fuego_v1"
ASSET_TILES = "projects/mapbiomas-colombia/assets/DATOS_AUXILIARES/VECTORES/SENTINEL/sentinel_2_index_shapefile"

GLOBAL_CONFIG = {
    "country": COUNTRY_SELECTED,
    "campaign": CAMPAIGN_SELECTED,
    "sensor": "sentinel2",
    "periodicity": "monthly",
    "mosaic_methods": ["minnbr", "minnbr_buffer"],
    "personal_task_flag": "MONITOR",
    "language": "en",  # en | es | pt | fr | id
    "gcs_bucket": "mapbiomas-fire",
    "gcs_library_images_prefix": f"sudamerica/{COUNTRY_SELECTED}/CATALOG_01",
    "gcs_campaigns_prefix": f"sudamerica/{COUNTRY_SELECTED}/CATALOG_01",
    "gee_project": GEE_PROJECT,
    "gee_library_images_prefix": f"projects/mapbiomas-{COUNTRY_SELECTED}/assets/FIRE/CATALOG_01",
    "gee_campaigns_prefix": f"projects/mapbiomas-{COUNTRY_SELECTED}/assets/FIRE/CATALOG_01",
    "asset_regions": ASSET_REGIONS,
}


In [ ]:
#@title 2. Preparar Colab y autenticar
import os
import shutil
import subprocess
import sys
from pathlib import Path
from importlib import import_module, invalidate_caches, reload

# Crear siempre un clon nuevo, incluso al repetir desde otro directorio.
os.chdir("/content")
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
    print("Clon anterior eliminado.")
subprocess.run(
    ["git", "clone", "--branch", REPO_BRANCH, REPO_URL, REPO_DIR],
    check=True,
)

subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(["apt-get", "install", "-y", "-qq", "gdal-bin", "python3-gdal"], check=True)
%pip install -q gdown "geopandas>=1.1" "pystac-client>=0.9" "rasterio>=1.4" "folium>=0.20" matplotlib tqdm planetary-computer earthengine-api geemap scikit-learn scipy gcsfs psutil google-api-python-client "ipywidgets>=8,<9" "jupyter-loopback[comm]>=0.3,<1"

CARPETA_MODULO = Path(REPO_DIR) / CORE_RELATIVE_PATH
MODULOS_REQUERIDOS = (
    "M0_auth_config", "M_mosaics", "M_lang", "M_gcs", "M_cache", "Sentinel2_pipeline_test",
    "M1_export_logic", "M_blocklist", "M1_mosaic_preview_ui",
    "M2_mosaic_logic", "M2_mosaic_ui", "M_ui_components", "M_scene_inspector", "M_cloud_mask_comparison", "M_resource_monitor",
)
faltantes = [f"{name}.py" for name in MODULOS_REQUERIDOS if not (CARPETA_MODULO / f"{name}.py").is_file()]
if faltantes:
    raise FileNotFoundError(f"Faltan archivos en {CARPETA_MODULO}: {', '.join(faltantes)}")
sys.path = [p for p in sys.path if Path(p or os.curdir).resolve() != CARPETA_MODULO.resolve()]
sys.path.insert(0, str(CARPETA_MODULO))
os.chdir(CARPETA_MODULO)
invalidate_caches()

# Recargar la configuración y sus dependencias después de renovar el clon.
for name in MODULOS_REQUERIDOS:
    sys.modules.pop(name, None)
from M0_auth_config import set_global_opts, authenticate, print_config

set_global_opts(**GLOBAL_CONFIG)
authenticate()
print_config()

NotebookWorkflow = reload(import_module("Sentinel2_pipeline_test")).NotebookWorkflow
workflow = NotebookWorkflow(Path(DIRECTORIO_TRABAJO))


## Almacenamiento de resultados en Drive

- Las regiones y teselas se leen de los assets GEE definidos en la configuración.
- Drive se usa únicamente para conservar resultados del flujo de mosaicos.


In [ ]:
#@title Carpeta de resultados en Drive
DRIVE_ROOT = "/content/drive/MyDrive/FuegoMonitor"
workflow.use_drive(DRIVE_ROOT)


## Mosaico mensual: vista y exportación

- Elija año, mes completo y método; pulse **Preparar y visualizar**.
- Composición M1: mínimo NBR, Cloud Score Plus ≥ 0.40 y buffer INPE opcional.
- Vista en geemap de toda Colombia; se exporta el mismo mosaico.
- Salida a 10 m: reflectancia en byte (×100) y día del año en int16.
- Elija **GEE Assets**, **GCS** o ambos y pulse **Exportar mosaico**.
- Se exportan las siete bandas, con una tarea por banda y destino. GCS guarda COG en `CHUNKS`.
- Pulse **Actualizar tareas** para consultar el estado.


In [ ]:
#@title Mosaico mensual y tareas de exportación
from M1_mosaic_preview_ui import run_ui, start_export

ui_exporter = run_ui(years=MOSAIC_YEARS, tiles_asset=ASSET_TILES)

# Alternativa al botón Exportar mosaico, después de preparar la vista:
# start_export(ui_exporter)


## [M2] Ensamblar mosaicos desde GCS

- Funciona en una sesión nueva, después de configurar y autenticar; no requiere ejecutar M1.
- Pulse **Consultar GCS**, seleccione los mosaicos disponibles y pulse **Procesar seleccionados**.
- Comprueba los nombres de `CHUNKS` y `COG` por país, sensor, método, mes y banda.
- Omite los COGs existentes y bloquea mosaicos incompletos o con exportaciones en curso.
- Los resultados quedan en `ui_assembler.results`.


In [ ]:
#@title M2 · Mosaicos pendientes en GCS
from M2_mosaic_ui import run_from_gcs

ui_assembler = run_from_gcs()


## OPCIONAL· Descargar escenas del mes por región fuera de GEE

- Elija región, año, mes y fuente en el menú y pulse **Descargar** para iniciar.
- Se consulta la blocklist y se imprimen las escenas excluidas; no se descargan.
- **Modo de prueba activado:** máximo 10 escenas permitidas. Desactivado: todas las disponibles en la fuente.
- Se descargan las 12 bandas espectrales L2A (B1–B9, B8A, B11 y B12), más SCL para CloudSEN12 y la comparación; sin filtro de nubes ni máscaras aplicadas.
- SCL permite enmascarar nubes y sombras; Cloud Score Plus (`cs`) de M1 es un producto separado y no está incluido.
- Las rutas quedan en `ui_monthly.result["result"].paths` cuando hay descargas.


In [ ]:
#@title Descargar escenas · región, año, mes y fuente
from M_scene_inspector import run_monthly_download_ui

ui_monthly = run_monthly_download_ui(
    workflow,
    regions_asset=GLOBAL_CONFIG["asset_regions"],
    tiles_asset=ASSET_TILES,
)


## Comparar máscaras de nubes y sombras

- Usa las escenas descargadas, recortadas a la región; no compone todavía un mosaico temporal.
- Modelo `cloudsen12l2a` para L2A, como en el [ejemplo CloudSEN12](https://github.com/IPL-UV/cloudsen12_models/blob/main/notebooks/run_in_gee_image.ipynb). Requiere las 12 bandas L2A y SCL; vuelva a descargar para completar escenas antiguas.
- Inferencia por bloques sobre la malla NIR de 10 m, con reflectancia calibrada. SCL se remuestrea con vecino más cercano.
- **Debug:** cuadrícula de 3 filas × 10 columnas: original SWIR1/NIR/rojo, máscara SCL aplicada y máscara CloudSEN12 aplicada. Negro indica píxeles excluidos. Si hay menos de 10 escenas, quedan columnas vacías.
- SCL excluye sombras (3), nubes (8/9) y cirros (10); CloudSEN12 excluye nubes gruesas, finas y sombras. Se guardan máscaras GeoTIFF; los originales se conservan.
- **Recursos:** monitor en vivo y comparación por etapa de tiempo, RAM del proceso y su incremento; en CUDA, VRAM PyTorch y utilización GPU del dispositivo. SCL se ejecuta en CPU.
- `USAR_GPU` activa CUDA para CloudSEN12 cuando está disponible. Preparación, carga del modelo y debug se miden por separado.
- Se guardan `resource_usage.json`, `resource_usage.csv` y `resource_comparison.csv`. Para repetir mediciones de máscaras en caché, active `SOBRESCRIBIR_MASCARAS`.


In [ ]:
#@title Procesar nubes y sombras · SCL vs CloudSEN12
USAR_GPU = True  #@param {type:"boolean"}
DEBUG_NUBES = True  #@param {type:"boolean"}
SOBRESCRIBIR_MASCARAS = False  #@param {type:"boolean"}

if ui_monthly.result is None or ui_monthly.result.get("result") is None:
    raise ValueError("Pulse Descargar en el menú anterior y espere a que termine.")

monthly_download = ui_monthly.result
download_parameters = ui_monthly.download_parameters

%pip install -q "cloudsen12_models==1.0.2" psutil nvidia-ml-py

from M_cloud_mask_comparison import process_downloaded_scenes

cloud_processing = process_downloaded_scenes(
    monthly_download,
    output_dir=workflow.root / "cloud_mask_comparison" / str(download_parameters["region_id"]) / download_parameters["start_date"][:7] / ui_monthly.download_source,
    region_path=ui_monthly.download_region_path,
    weights_dir=workflow.root / "cloudsen12_models",
    debug=DEBUG_NUBES,
    overwrite=SOBRESCRIBIR_MASCARAS,
    use_gpu=USAR_GPU,
    live_monitor=True,
)


## Flujo anterior opcional: mosaico regional con máscara SCL

- Las siguientes celdas conservan la vista y composición anteriores, con sus propios filtros.
- No utilizan la selección de escenas del panel anterior.
- La descarga y composición están desactivadas por defecto.


In [ ]:
#@title Parámetros del mosaico anterior (opcional)
INSPECTION_PARAMETERS = ui_monthly.parameters()

LEGACY_PARAMETERS = {
    "project": GLOBAL_CONFIG["gee_project"],
    "regions_asset": INSPECTION_PARAMETERS["regions_asset"],
    "tiles_asset": ASSET_TILES,
    "region_id": INSPECTION_PARAMETERS["region_id"],
    "start_date": INSPECTION_PARAMETERS["start_date"],
    "end_date": INSPECTION_PARAMETERS["end_date"],
    "max_cloud_cover": 20,
    "bands": ["B11", "B8", "B4", "B12"],  # amplíe esta lista para descargar más bandas
    "resolution_m": 20,  # resolución del mosaico descargado
    "scl_exclude": [0, 1, 2, 3, 8, 9, 10, 11],
    "vis": {"bands": ["swir1", "nir", "red"], "min": 0.03, "max": 0.4},
}
# Rojo, NIR, SWIR1, SWIR2 se incluyen siempre para los índices; SCL es auxiliar.


In [ ]:
workflow.load_gee_vectors(LEGACY_PARAMETERS)

MOSTRAR_ESTADISTICAS_GEE = True  #@param {type:"boolean"}

Map = workflow.preview(LEGACY_PARAMETERS, include_metadata=MOSTRAR_ESTADISTICAS_GEE)
Map


## 5. Descargar y componer

Active `DESCARGAR_IMAGENES`. Se reutiliza el mosaico guardado para las mismas fechas, región, fuente y parámetros.
`SOBRESCRIBIR_MOSAICO = True` fuerza una nueva descarga y composición.

Se procesa temporalmente y se guarda el COG, metadatos y tiempos en `DRIVE_ROOT/mosaics/AAAA-MM`.
`CONSERVAR_IMAGENES = False` elimina las escenas tras procesarlas.
Los tiempos de descarga y procesamiento se solapan.


In [ ]:
SOURCE = "planetary-computer"  #@param ["planetary-computer", "earth-search"]
DESCARGAR_IMAGENES = False  #@param {type:"boolean"}
CONSERVAR_IMAGENES = False  #@param {type:"boolean"}

SOBRESCRIBIR_MOSAICO = False  #@param {type:"boolean"}

composicion = workflow.download(
    LEGACY_PARAMETERS, source=SOURCE, enabled=DESCARGAR_IMAGENES,
    keep_inputs=CONSERVAR_IMAGENES, overwrite=SOBRESCRIBIR_MOSAICO,
)


## 6. Comparación pausada

La comparación está comentada. Continúe en la sección 7 para abrir el mosaico y tomar muestras.


In [ ]:
# VIEW = "color"  #@param ["color", "NBR"]
# RENOVAR_TESELAS_GEE = False  #@param {type:"boolean"}

# comparison = workflow.compare(
#     LEGACY_PARAMETERS, source=SOURCE, view=VIEW, refresh_gee=RENOVAR_TESELAS_GEE,
# )
# comparison

## 7. Muestras, modelos y clasificación

Edite muestras en Folium y elija el archivo para entrenar.
Seleccione **Crear modelo nuevo** y el algoritmo, o aplique/reentrene uno guardado.
Los resultados quedan como capas independientes; el panel muestra tiempo y RAM.


In [ ]:
MODEL_PARAMETERS = {
    "gaussian_nb": {"var_smoothing": 1e-9},
    "logistic_regression": {"C": 1.0, "max_iter": 1000},
    "svm": {"C": 1.0, "gamma": "scale"},
    "extra_trees": {},
    "gradient_boosting": {"learning_rate": 0.1, "max_depth": 3},
}
RF_PARAMETERS = {
    "classifier_parameters": MODEL_PARAMETERS,
    "n_estimators": 200, "max_depth": 20,
    "min_samples_leaf": 2,
    "max_samples_per_class": 5000, "min_samples_per_class": 10,
    "random_state": 42, "n_jobs": 2, "block_size": 512,
}


In [ ]:
MOSAIC_PATH = ""  # vacío: recuperar el mosaico de LEGACY_PARAMETERS
SOURCE = globals().get("SOURCE", "planetary-computer")

training = workflow.training_canvas(
    LEGACY_PARAMETERS, source=SOURCE, mosaic_path=MOSAIC_PATH or None, rf_parameters=RF_PARAMETERS,
)
training.show()


## Fuentes Sentinel-2 L2A

Reflectancia de superficie ortorrectificada y corregida atmosféricamente.
Cuotas consultadas el **19 de octubre de 2026**.

| Fuente | Cuota gratuita | Requisitos |
| --- | --- | --- |
| [Earth Search / AWS — COG de Nivel 2A](https://registry.opendata.aws/sentinel-2-l2a-cogs/) **(opción de descarga)** | Acceso gratuito; la documentación no especifica una cuota mensual en GB. | Ninguno para descargar COG públicos por HTTPS. |
| [Copernicus Data Space — productos de Nivel 2A](https://documentation.dataspace.copernicus.eu/Data/SentinelMissions/Sentinel2.html#sentinel-2-level-2a-surface-reflectance) | [12 TB por cada período móvil de 30 días a velocidad normal; después se reduce la velocidad.](https://documentation.dataspace.copernicus.eu/Quotas.html) | Cuenta gratuita de CDSE; token de acceso o credenciales S3. |
| [CDSE Sentinel Hub — API de procesamiento L2A](https://documentation.dataspace.copernicus.eu/APIs/SentinelHub/Data/S2L2A.html) | [10.000 unidades de procesamiento y 10.000 solicitudes al mes.](https://documentation.dataspace.copernicus.eu/Quotas.html) | Cuenta CDSE y cliente OAuth; solicitar bandas espectrales en unidades REFLECTANCE. |
| [Microsoft Planetary Computer — sentinel-2-l2a](https://planetarycomputer.microsoft.com/dataset/sentinel-2-l2a) | [Acceso gratuito; no se especifica una cuota mensual en GB; hay límites de frecuencia al solicitar tokens.](https://planetarycomputer.microsoft.com/docs/concepts/sas/) | URL firmadas con SAS; se pueden obtener tokens de forma anónima. |
| [Google Earth Engine — S2_SR_HARMONIZED](https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_SR_HARMONIZED) | [Nivel Comunidad: 150 horas-EECU por proyecto y mes; rendimiento reducido al superar la cuota.](https://developers.google.com/earth-engine/guides/noncommercial_tiers) | Cuenta Google, proyecto de Cloud registrado, uso no comercial verificado y autenticación. |

Descarga: Earth Search o Planetary Computer. Inspección: GEE.
Las cuotas de descarga y cómputo son distintas; una cuota no publicada no implica acceso ilimitado.